# Recipe: Answee Q&A Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/knivok/ai-playbook/blob/main/notebooks/03_knovik_recipes/01_answee_qa_agent.ipynb)

**Stack:** LlamaIndex (retrieval) + LangGraph (agent loop)  
**Product:** Answee (answee.com)  
**Pattern:** RAG agent with memory and follow-up handling

---

## What this builds

Answee's core feature: a user asks a question about a product category (e.g., "What's the best Zigbee hub for Apple HomeKit?"), the agent:

1. Retrieves relevant product documents from the knowledge base
2. Reasons over them to construct a ranked answer
3. Handles follow-up questions with memory of the conversation
4. Cites sources for every claim

## Architecture

```
User question
     ↓
LlamaIndex VectorStoreIndex (ChromaDB)
     ↓ retrieves top-k chunks
LangGraph agent
     ↓ reasons + synthesizes
Answer with citations + follow-up suggestions
```

In [ ]:
!pip install -q llama-index llama-index-llms-anthropic llama-index-embeddings-openai langgraph langchain-anthropic chromadb rich

In [ ]:
import os
from getpass import getpass
from typing import TypedDict, List
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, END, START
from llama_index.core import VectorStoreIndex, Document
from llama_index.llms.anthropic import Anthropic as LlamaAnthropic
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

console = Console()
print("✅ Setup complete")

In [ ]:
# ── Sample product knowledge base (in production: load from your Strapi CMS) ──

PRODUCT_DOCS = [
    Document(text="""
Product: Aqara M2 Hub
Price: $49.99
Protocols: Zigbee 3.0, HomeKit, Matter
Compatibility: Apple HomeKit, Google Home, Amazon Alexa
Key features: Local processing, no cloud dependency, supports 128 Zigbee devices
Best for: Apple HomeKit users who want Zigbee device support
Rating: 4.6/5 (2,400 reviews)
ATHBridge URL: https://athbridge.com/products/aqara-m2
"""),
    Document(text="""
Product: Sonoff Zigbee Bridge Pro
Price: $19.99
Protocols: Zigbee 3.0
Compatibility: Requires eWeLink app, limited HomeKit support via workaround
Key features: Supports 26 sub-devices, local mode available
Best for: Budget-conscious users, eWeLink ecosystem
Rating: 4.3/5 (8,900 reviews)
ATHBridge URL: https://athbridge.com/products/sonoff-bridge-pro
"""),
    Document(text="""
Product: Home Assistant SkyConnect
Price: $29.99
Protocols: Zigbee, Thread, Matter
Compatibility: Home Assistant only (open source)
Key features: USB stick, no hub required, full local control, developer-friendly
Best for: Technical users who want maximum control, Home Assistant ecosystem
Rating: 4.7/5 (1,200 reviews)
ATHBridge URL: https://athbridge.com/products/skyconnect
"""),
]

# Build the LlamaIndex
index = VectorStoreIndex.from_documents(
    PRODUCT_DOCS,
    llm=LlamaAnthropic(model="claude-3-5-sonnet-20241022")
)
query_engine = index.as_query_engine(similarity_top_k=3)

print("✅ Knowledge base indexed with", len(PRODUCT_DOCS), "products")

In [ ]:
# ── Agent State ───────────────────────────────────────────────────────────────

class AnsweeState(TypedDict):
    question: str
    conversation_history: List[dict]
    retrieved_context: str
    answer: str
    follow_up_suggestions: List[str]


# ── Nodes ─────────────────────────────────────────────────────────────────────

def retrieve_node(state: AnsweeState) -> dict:
    """Retrieve relevant product information from the knowledge base."""
    console.print(f"\n[cyan]🔍 Retrieving for:[/] {state['question']}")
    response = query_engine.query(state["question"])
    return {"retrieved_context": str(response)}


llm = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)

def answer_node(state: AnsweeState) -> dict:
    """Generate a cited answer using retrieved context."""
    console.print("[green]💬 Generating answer...[/]")

    history_text = ""
    for turn in state.get("conversation_history", []):
        history_text += f"User: {turn['user']}\nAnswee: {turn['assistant']}\n\n"

    response = llm.invoke([
        SystemMessage(content="""You are Answee, a helpful product recommendation assistant.
Always cite specific products and prices. Be direct and helpful.
End with 2-3 follow-up questions the user might have, formatted as a JSON list under the key 'follow_ups'."""),
        HumanMessage(content=f"""
Conversation so far:
{history_text}

Product knowledge:
{state['retrieved_context']}

Current question: {state['question']}

Answer the question. Then add:
FOLLOW_UPS: ["question1", "question2", "question3"]
""")
    ])

    import re, json
    content = response.content
    follow_ups = []

    match = re.search(r'FOLLOW_UPS:\s*(\[.*?\])', content, re.DOTALL)
    if match:
        follow_ups = json.loads(match.group(1))
        content = content[:match.start()].strip()

    return {"answer": content, "follow_up_suggestions": follow_ups}


# ── Graph ─────────────────────────────────────────────────────────────────────

workflow = StateGraph(AnsweeState)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("answer", answer_node)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "answer")
workflow.add_edge("answer", END)
answee_agent = workflow.compile()

print("✅ Answee agent ready")

In [ ]:
# ── Multi-turn conversation demo ──────────────────────────────────────────────

history = []

def ask_answee(question: str):
    result = answee_agent.invoke({
        "question": question,
        "conversation_history": history,
        "retrieved_context": "",
        "answer": "",
        "follow_up_suggestions": []
    })

    console.print(Panel(
        Markdown(result["answer"]),
        title=f"💬 {question}",
        border_style="green"
    ))

    if result["follow_up_suggestions"]:
        console.print("[dim]You might also ask:[/]")
        for q in result["follow_up_suggestions"]:
            console.print(f"  • {q}")

    history.append({"user": question, "assistant": result["answer"]})
    return result


# Turn 1
ask_answee("What's the best Zigbee hub for Apple HomeKit users?")

In [ ]:
# Turn 2 - follow-up with conversation memory
ask_answee("What about for someone on a tight budget?")

## Production notes for Answee

1. **Replace `PRODUCT_DOCS`** with a LlamaIndex loader that pulls from your Strapi CMS or ATHBridge WooCommerce product catalog
2. **Use ChromaDB** instead of the in-memory index for persistence across server restarts
3. **Add a re-ranking node** between `retrieve` and `answer` for better relevance on large catalogs
4. **Persist conversation history** in Redis or PostgreSQL keyed by session ID
5. **Track citations** - capture `source_nodes` from LlamaIndex response to build attribution links back to ATHBridge product pages
